In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"


## Reasoning(OpenMath) and Non-reasoning(Finetome) reasoning

In [2]:
import torch
import gc
import re
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics import accuracy_score, f1_score
from rouge_score import rouge_scorer
from tqdm import tqdm
from unsloth import FastLanguageModel
# ─────────────────────────────────────────────
# OPENMATHREASONING EVALUATION
# ─────────────────────────────────────────────

def extract_math_answer(text: str) -> str:
    """Extract final answer — tries \\boxed{} first, then last number."""
    # Try \boxed{...}
    boxed = re.findall(r"\\boxed\{([^}]+)\}", text)
    if boxed:
        return boxed[-1].strip()
    # Try #### answer format (common in math datasets)
    hash_match = re.findall(r"####\s*([^\n]+)", text)
    if hash_match:
        return hash_match[-1].strip()
    # Fall back to last number in text
    numbers = re.findall(r"-?\d+\.?\d*", text)
    if numbers:
        return numbers[-1].strip()
    return text.strip()


def format_math_prompt(problem: str) -> str:
    return (
        f"Solve the following math problem. "
        f"Show your reasoning and put your final answer in \\boxed{{}}.\n\n"
        f"Problem: {problem}\n\n"
        f"Solution:"
    )


def evaluate_math(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples: int = 200,
    batch_size: int = 4,
    max_new_tokens: int = 256,
    device: str = "cuda",
) -> dict:
    """Evaluate on OpenMathReasoning-mini using exact match on final answer."""
    print(f"\n{'─'*60}")
    print(f"[Math] Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()

    dataset = load_dataset("unsloth/OpenMathReasoning-mini", split="cot")
    dataset = dataset.select(range(min(num_samples, len(dataset))))

    # Check column names
    print(f"Columns: {dataset.column_names}")

    preds  = []
    labels = []

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [math]"):
        batch = dataset[i : i + batch_size]

        # OpenMathReasoning columns: problem, solution, answer
        problems       = batch["problem"]
        true_answers   = [extract_math_answer(a) for a in batch["expected_answer"]]

        prompts = [format_math_prompt(p) for p in problems]

        inputs = tokenizer(
            prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens     = max_new_tokens,
                do_sample          = False,
                pad_token_id       = tokenizer.eos_token_id,
                eos_token_id       = tokenizer.eos_token_id,
                repetition_penalty = 1.3,
            )

        for j, output in enumerate(outputs):
            input_len   = inputs["input_ids"].shape[1]
            generated   = tokenizer.decode(output[input_len:], skip_special_tokens=True)
            pred_answer = extract_math_answer(generated)
            preds.append(pred_answer)
            labels.append(true_answers[j])

    # Exact match
    exact_matches = [p.strip() == l.strip() for p, l in zip(preds, labels)]
    exact_match   = round(sum(exact_matches) / len(exact_matches), 4)

    result = {
        "repo_id"      : model_name,
        "exact_match"  : exact_match,
        "num_samples"  : num_samples,
        "sample_preds" : list(zip(labels[:5], preds[:5])),  # first 5 for inspection
    }

    print(f"  Exact Match: {exact_match:.4f}")
    print(f"  Sample predictions (true → pred):")
    for true, pred in result["sample_preds"]:
        print(f"    {true:<20} → {pred}")

    return result


# ─────────────────────────────────────────────
# FINETOME EVALUATION (ROUGE)
# ─────────────────────────────────────────────

def format_finetome_prompt(conversation: list) -> tuple[str, str]:
    """
    Extract user prompt and reference response from conversation turns.
    FineTome-100k has a 'conversations' field with role/value pairs.
    Returns (prompt, reference_response).
    """
    prompt    = ""
    reference = ""

    for turn in conversation:
        role  = turn.get("from", turn.get("role", ""))
        value = turn.get("value", turn.get("content", ""))
        if role in ("human", "user") and not prompt:
            prompt = f"User: {value}\n\nAssistant:"
        elif role in ("gpt", "assistant") and not reference:
            reference = value

    return prompt, reference


def evaluate_finetome(
    model,
    tokenizer,
    model_name: str = "model",
    num_samples: int = 200,
    batch_size: int = 4,
    max_new_tokens: int = 256,
    device: str = "cuda",
) -> dict:
    """Evaluate on FineTome-100k using ROUGE-1, ROUGE-2, ROUGE-L."""
    print(f"\n{'─'*60}")
    print(f"[FineTome] Evaluating: {model_name}")
    print(f"{'─'*60}")

    model.eval()

    dataset = load_dataset("mlabonne/FineTome-100k", split="train")
    dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))

    print(f"Columns: {dataset.column_names}")

    scorer    = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    all_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

    for i in tqdm(range(0, len(dataset), batch_size), desc=f"{model_name} [finetome]"):
        batch      = dataset[i : i + batch_size]
        prompts    = []
        references = []

        for conv in batch["conversations"]:
            prompt, reference = format_finetome_prompt(conv)
            prompts.append(prompt)
            references.append(reference)

        # Skip if no valid prompts extracted
        if not any(prompts):
            continue

        inputs = tokenizer(
            prompts,
            return_tensors = "pt",
            padding        = True,
            truncation     = True,
            max_length     = 512,
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens     = max_new_tokens,
                do_sample          = False,
                pad_token_id       = tokenizer.eos_token_id,
                eos_token_id       = tokenizer.eos_token_id,
                repetition_penalty = 1.3,
            )

        for j, output in enumerate(outputs):
            input_len = inputs["input_ids"].shape[1]
            generated = tokenizer.decode(output[input_len:], skip_special_tokens=True)

            if references[j]:
                scores = scorer.score(references[j], generated)
                all_scores["rouge1"].append(scores["rouge1"].fmeasure)
                all_scores["rouge2"].append(scores["rouge2"].fmeasure)
                all_scores["rougeL"].append(scores["rougeL"].fmeasure)

    result = {
        "repo_id"    : model_name,
        "rouge1"     : round(sum(all_scores["rouge1"]) / len(all_scores["rouge1"]), 4),
        "rouge2"     : round(sum(all_scores["rouge2"]) / len(all_scores["rouge2"]), 4),
        "rougeL"     : round(sum(all_scores["rougeL"]) / len(all_scores["rougeL"]), 4),
        "num_samples": num_samples,
    }

    print(f"  ROUGE-1: {result['rouge1']:.4f}")
    print(f"  ROUGE-2: {result['rouge2']:.4f}")
    print(f"  ROUGE-L: {result['rougeL']:.4f}")

    return result


# ─────────────────────────────────────────────
# EVALUATE ALL MERGED MODELS
# ─────────────────────────────────────────────

def evaluate_all_qwen_models(
    repos: list[str],
    num_samples: int = 200,
    batch_size: int = 4,
    device: str = "cuda",
) -> dict:
    all_results = {}

    for repo in repos:
        print(f"\n{'═'*60}")
        print(f"Model: {repo}")
        print(f"{'═'*60}")

        # ← use Unsloth instead of AutoModelForCausalLM
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name     = repo,
            max_seq_length = 1024,
            load_in_4bit   = True,
            dtype          = torch.float16,
        )
        FastLanguageModel.for_inference(model)  # ← required for fast generation

        math_result     = evaluate_math(model, tokenizer, model_name=repo,
                                        num_samples=num_samples, batch_size=batch_size)
        finetome_result = evaluate_finetome(model, tokenizer, model_name=repo,
                                            num_samples=num_samples, batch_size=batch_size)

        all_results[repo] = {
            "math"    : math_result,
            "finetome": finetome_result,
        }

        del model, tokenizer
        gc.collect()
        torch.cuda.empty_cache()

    # ── Summary table ──
    print(f"\n{'═'*70}")
    print(f"{'MODEL':<35} {'EXACT':>7} {'R1':>7} {'R2':>7} {'RL':>7}")
    print(f"{'─'*70}")
    for repo, r in all_results.items():
        name = repo.split("/")[-1]
        print(
            f"{name:<35} "
            f"{r['math']['exact_match']:>7.4f} "
            f"{r['finetome']['rouge1']:>7.4f} "
            f"{r['finetome']['rouge2']:>7.4f} "
            f"{r['finetome']['rougeL']:>7.4f}"
        )
    print(f"{'═'*70}")

    return all_results
# ── Usage ──

repos = [
    "Srishtik/Qwen3-0.6B-linear-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-svd-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-ties-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-dare-3-adapters-merged",
    "Srishtik/Qwen3-0.6B-slerp-3-adapters-merged",
]

all_results = evaluate_all_qwen_models(
    repos       = repos,
    num_samples = 200,
    batch_size  = 4,
)

/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:153: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!

════════════════════════════════════════════════════════════
Model: Srishtik/Qwen3-0.6B-linear-3-adapters-merged
════════════════════════════════════════════════════════════
==((====))==  Unsloth 2026.6.7: Fast Qwen3 patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

...


────────────────────────────────────────────────────────────
[Math] Evaluating: Srishtik/Qwen3-0.6B-slerp-3-adapters-merged
────────────────────────────────────────────────────────────
Columns: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode']


Srishtik/Qwen3-0.6B-slerp-3-adapters-merged [math]: 100%|██████████| 50/50 [13:07<00:00, 15.74s/it]


  Exact Match: 0.0900
  Sample predictions (true → pred):
    14                   → 52
    convergent           → 1
    2                    → 0
    \(+\infty\)          → 5
    1                    → 2

────────────────────────────────────────────────────────────
[FineTome] Evaluating: Srishtik/Qwen3-0.6B-slerp-3-adapters-merged
────────────────────────────────────────────────────────────
Columns: ['conversations', 'source', 'score']


Srishtik/Qwen3-0.6B-slerp-3-adapters-merged [finetome]: 100%|██████████| 50/50 [13:13<00:00, 15.87s/it]


  ROUGE-1: 0.3160
  ROUGE-2: 0.0676
  ROUGE-L: 0.1529

══════════════════════════════════════════════════════════════════════
MODEL                                 EXACT      R1      R2      RL
──────────────────────────────────────────────────────────────────────
Qwen3-0.6B-linear-3-adapters-merged  0.1200  0.3088  0.0623  0.1465
Qwen3-0.6B-svd-3-adapters-merged     0.1400  0.3090  0.0626  0.1488
Qwen3-0.6B-ties-3-adapters-merged    0.0900  0.3294  0.0739  0.1595
Qwen3-0.6B-dare-3-adapters-merged    0.1050  0.3264  0.0721  0.1578
Qwen3-0.6B-slerp-3-adapters-merged   0.0900  0.3160  0.0676  0.1529
══════════════════════════════════════════════════════════════════════
